# 03_0. Natural-disaster rule and validation halves / Правило стихийных бедствий и половины validation

**EN.** Stage 3 starts with two things every later notebook depends on, plus a re-check. There is one task, Y1: a default (90+ days past due or a loss liquidation) within 12 months among loans that are current at t and have never been in default before.

1. **Disaster rule.** In 2017 hurricanes Harvey, Irma and Maria pushed defaults up in the label windows of part of validation. Stage 2 removed them with a simple flag (TX, FL, PR, VI at two dates). Now a formal rule replaces it and is applied the same way to validation, test and stress scenarios: an observation (loan, date t) is excluded if a FEMA **Major Disaster** with assistance to individuals **began** between 3 months before t and 12 months after t, and the loan's 3-digit ZIP code (ZIP3) is **affected** (at least half of its housing units are in declared counties). The rule looks only at external events, never at the loan's outcome, so it cannot "clean" the sample in favour of a model.
2. **Validation halves A / B.** Validation loans are split in two halves: **A** for tuning hyperparameters and early stopping, **B** for calibration. A loan's snapshots stay in one half.
3. **Re-check of stage 2.** Stage-2 conclusions (the selected 18 features are not worse than all 59 candidates; the geography group is removed) were measured on validation without the old hurricane flag. We repeat them with the new rule.

**RU.** Этап 3 начинается с двух вещей, от которых зависят все следующие ноутбуки, и одной перепроверки. Задача одна — Y1: дефолт (просрочка 90+ дней или ликвидация с убытком) в течение 12 месяцев у кредитов, которые на дату t платят вовремя и раньше в дефолте не были.

1. **Правило стихийных бедствий.** В 2017 г. ураганы Харви, Ирма и Мария подняли дефолты в окнах меток части validation. На этапе 2 их убирал простой флаг (TX, FL, PR, VI на двух датах). Теперь его заменяет формальное правило, одинаковое для validation, test и стресс-сценариев: наблюдение (кредит, дата t) исключается, если **крупное бедствие** FEMA (Major Disaster) с помощью населению **началось** в промежутке от 3 месяцев до t до 12 месяцев после t, и 3-значный ZIP-код объекта (ZIP3) **затронут** (не меньше половины его жилья — в объявленных округах). Правило смотрит только на внешнее событие, а не на исход кредита, поэтому не может «подчистить» выборку в пользу модели.
2. **Половины validation A / B.** Кредиты validation делятся на две половины: **A** — подбор гиперпараметров и ранняя остановка, **B** — калибровка. Все снимки одного кредита попадают в одну половину.
3. **Перепроверка этапа 2.** Выводы этапа 2 (18 отобранных признаков не хуже всех 59 кандидатов; группа географии исключена) считались на validation без старого флага урагана. Повторяем их с новым правилом.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})
pl.Config.set_tbl_rows(30)

from xai_ews.config import feature_config
from xai_ews.data.disasters import FEMA_FILE, build_flags, exclusion_expr
from xai_ews.data.features import candidate_features
from xai_ews.data.panel import scan_panel
from xai_ews.features.evaluation import compare_models
from xai_ews.features.importance import fit_predict_validation
from xai_ews.models.data import exclusion_flags, make_validation_halves, with_disaster_flags
from xai_ews.paths import FEATURE_SELECTION_DIR, MODELING_DIR, VALIDATION_HALVES

MODELING_DIR.mkdir(parents=True, exist_ok=True)
FCFG = feature_config()
SEED = FCFG["data"]["seed"]

### 1. FEMA declarations / Декларации FEMA

**EN.** OpenFEMA lists every declared county of every disaster. We keep Major Disasters (`DR`) with assistance to individuals. The rule was first written with the field `iaProgramDeclared`, but the table below shows that this **legacy flag is never set after 2008**: Individual Assistance was replaced by the Individuals and Households Program (`ihProgramDeclared`). Taken literally, the rule would exclude nothing after 2008, the 2017 hurricanes included, so the rule uses "IA **or** IH".

**RU.** OpenFEMA перечисляет все объявленные округа каждого бедствия. Берём крупные бедствия (`DR`) с помощью населению. Сначала в правиле было указано поле `iaProgramDeclared`, но таблица ниже показывает, что этот **устаревший флаг после 2008 г. не ставится**: программу Individual Assistance заменила программа Individuals and Households (`ihProgramDeclared`). Буквально правило не исключило бы ничего после 2008 г., включая ураганы 2017 г., поэтому правило использует «IA **или** IH».

In [2]:
raw = pl.read_csv(FEMA_FILE, infer_schema=False)
dr = raw.filter(pl.col("declarationType") == "DR").with_columns(
    pl.col("incidentBeginDate").str.slice(0, 4).cast(pl.Int32).alias("year"))
by_year = (dr.filter(pl.col("year").is_between(2006, 2025)).group_by("year").agg(
    pl.col("disasterNumber").n_unique().alias("DR"),
    pl.col("disasterNumber").filter(pl.col("iaProgramDeclared") == "1").n_unique().alias("with_IA"),
    pl.col("disasterNumber").filter(pl.col("ihProgramDeclared") == "1").n_unique().alias("with_IH"),
    pl.col("disasterNumber").filter((pl.col("iaProgramDeclared") == "1") | (pl.col("ihProgramDeclared") == "1"))
    .n_unique().alias("used")).sort("year"))
by_year.to_pandas().set_index("year").T

year,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
DR,49,59,79,63,74,97,48,63,43,52,36,62,61,57,113,58,48,71,94,50
with_IA,24,32,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
with_IH,24,31,36,20,18,38,18,9,7,14,15,18,17,13,78,24,16,28,48,25
used,24,32,36,20,18,38,18,9,7,14,15,18,17,13,78,24,16,28,48,25


In [3]:
res = build_flags()          # writes data/processed/disaster_flags.parquet and the manifest rows
decl = res["declarations"].filter(pl.col("year").is_between(2006, 2025))
types = decl.group_by("incident_type").agg(pl.col("disaster").n_unique().alias("disasters"),
                                           pl.len().alias("county_rows")).sort("disasters", descending=True)
print("disasters 2006-2025 used by the rule:", decl["disaster"].n_unique(),
      "| tribal designations without a county (not mapped):", int(decl["tribal"].sum()))
types.to_pandas()

disasters 2006-2025 used by the rule: 498 | tribal designations without a county (not mapped): 72


,incident_type,disasters,county_rows
0,Severe Storm,212,2517
1,Flood,78,786
2,Hurricane,73,1210
3,Biological,58,3274
4,Fire,27,131
5,Tornado,19,137
6,Tropical Storm,8,97
7,Earthquake,7,57
8,Typhoon,3,8
9,Severe Ice Storm,3,169


### 2. Housing units by ZIP3 and county / Жильё по ZIP3 и округам

**EN.** The HUD ZIP–County crosswalk planned at first cannot be downloaded from outside the US (the site returns 403), so the official **Census 2010 ZCTA–county relationship file** is used: the number of housing units of every ZCTA (the Census approximation of a residential ZIP code) in every county. Housing units are summed to ZIP3; the share of a ZIP3 in a county is the county's part of the ZIP3's housing. Below: coverage of the file, Puerto Rico and the Virgin Islands, and FEMA counties that the 2010 file does not know.

**RU.** Сначала планировавшийся файл HUD ZIP–County недоступен не из США (сайт отвечает 403), поэтому используется официальный **файл Census 2010 ZCTA–county relationship**: число жилых единиц каждой ZCTA (переписное приближение жилого ZIP-кода) в каждом округе. Жильё суммируется до ZIP3; доля ZIP3 в округе — часть жилья ZIP3 в этом округе. Ниже: покрытие файла, Пуэрто-Рико и Виргинские острова, а также округа FEMA, которых нет в файле 2010 г.

In [4]:
sh = res["shares"]
cov = pd.DataFrame({
    "zip3": [sh["zip3"].n_unique()], "counties": [sh["county"].n_unique()],
    "puerto_rico_counties": [sh.filter(pl.col("county").str.starts_with("72"))["county"].n_unique()],
    "puerto_rico_zip3": [", ".join(sorted(sh.filter(pl.col("county").str.starts_with("72"))["zip3"].unique()))],
    "virgin_islands_counties": [sh.filter(pl.col("county").str.starts_with("78"))["county"].n_unique()]})
display(cov)
panel_zip3 = scan_panel().select("zip_code_short", "property_state").unique().collect()
missing = panel_zip3.join(sh.select(pl.col("zip3").alias("zip_code_short")).unique(), on="zip_code_short", how="anti")
print("panel ZIP3 without housing data:", missing.sort("zip_code_short").to_pandas().to_dict("records"))
print("FEMA county rows not in the 2010 file, by state:")
res["unmatched_counties"].group_by("state").agg(pl.len().alias("rows"), pl.col("year").min().alias("first_year"),
                                                pl.col("year").max().alias("last_year")).sort("rows", descending=True).to_pandas()

,zip3,counties,puerto_rico_counties,puerto_rico_zip3,virgin_islands_counties
0,894,3221,78,"006, 007, 009",0


panel ZIP3 without housing data: [{'zip_code_short': '000', 'property_state': 'OH'}, {'zip_code_short': '001', 'property_state': 'MA'}, {'zip_code_short': '008', 'property_state': 'PR'}, {'zip_code_short': '008', 'property_state': 'VI'}, {'zip_code_short': '094', 'property_state': 'CA'}, {'zip_code_short': '099', 'property_state': 'NJ'}, {'zip_code_short': '192', 'property_state': 'PA'}, {'zip_code_short': '340', 'property_state': 'FL'}, {'zip_code_short': '345', 'property_state': 'FL'}, {'zip_code_short': '459', 'property_state': 'OH'}, {'zip_code_short': '552', 'property_state': 'MN'}, {'zip_code_short': '682', 'property_state': 'KS'}, {'zip_code_short': '771', 'property_state': 'NJ'}, {'zip_code_short': '849', 'property_state': 'UT'}, {'zip_code_short': '861', 'property_state': 'CT'}, {'zip_code_short': '867', 'property_state': 'AZ'}, {'zip_code_short': '942', 'property_state': 'CA'}, {'zip_code_short': '965', 'property_state': 'WA'}, {'zip_code_short': '969', 'property_state': 'CA'

,state,rows,first_year,last_year
0,AS,55,1966,2020
1,MH,23,1988,1992
2,FM,10,1979,2004
3,AK,4,2020,2023


**EN. How to read.** The file covers 894 ZIP3 and 3,221 counties, including all 78 municipios of **Puerto Rico** (ZIP3 006, 007, 009). The **Virgin Islands** (ZIP3 008) and **Guam** (969) have no ZCTAs: for them the ZIP3 counts as affected whenever any of their counties is declared. Panel ZIP3 without housing data are, apart from VI and Guam, PO-box and military ZIP3 (001, 099, 340, ...): 5,041 of 32.2 M validation, test and stress rows, of which 4,772 are VI and Guam; the rest (269 rows) are never excluded. FEMA counties missing from the 2010 file are in American Samoa and the Pacific freely associated states (not in the panel) and 4 Alaska rows of 2020–2023 (borough changes after 2010). The 72 tribal designations have no county code and are not mapped.

**RU. Как читать.** Файл покрывает 894 ZIP3 и 3 221 округ, включая все 78 муниципалитетов **Пуэрто-Рико** (ZIP3 006, 007, 009). У **Виргинских островов** (ZIP3 008) и **Гуама** (969) ZCTA нет: для них ZIP3 считается затронутым, если объявлен любой их округ. Остальные ZIP3 панели без данных о жилье — абонентские ящики и военная почта (001, 099, 340 и др.): 5 041 из 32,2 млн строк validation, test и стресс-сценариев, из них 4 772 — Виргинские острова и Гуам; оставшиеся 269 строк никогда не исключаются. Округа FEMA, которых нет в файле 2010 г., — Американское Самоа и тихоокеанские ассоциированные государства (их нет в панели) и 4 строки Аляски 2020–2023 гг. (изменения боро после 2010 г.). 72 территории племён не имеют кода округа и не сопоставляются.

### 3. How many observations the rule removes / Сколько наблюдений исключает правило

**EN.** For each sample (Y1 population): the share of rows excluded by the main rule and by the three sensitivity variants (`hurricane` = hurricanes, tropical storms and typhoons only; `any_overlap` = a ZIP3 counts as affected if any part of it is in a declared county), and the event rates of kept and excluded rows. Events are counted per observation and per unique loan. Train is shown for information only: the rule never touches training data.

**RU.** Для каждой выборки (популяция Y1): доля наблюдений, исключённых основным правилом и тремя вариантами чувствительности (`hurricane` — только ураганы, тропические штормы и тайфуны; `any_overlap` — ZIP3 затронут при любом пересечении с объявленным округом), и доля событий среди оставшихся и исключённых. События считаются по наблюдениям и по уникальным кредитам. Train показан справочно: правило к обучающим данным не применяется.

In [5]:
lf = with_disaster_flags(scan_panel().filter(pl.col("split_main").is_not_null() | pl.col("stress").is_not_null())
                         .select("loan_identifier", "m", "zip_code_short", "stress", "split_main", "y1", "pop_y1"))
lf = lf.with_columns(pl.coalesce("split_main", "stress").alias("sample"),
                     *[exclusion_expr(v).alias(v) for v in ("main", "hurricane", "any_overlap")])
rows = []
for task in ("y1",):
    t = (lf.filter(pl.col(f"pop_{task}") & pl.col(task).is_not_null()).group_by("sample").agg(
        pl.len().alias("rows"),
        pl.col("main").mean().alias("excl_main"), pl.col("hurricane").mean().alias("excl_hurricane"),
        pl.col("any_overlap").mean().alias("excl_any_overlap"),
        pl.col(task).filter(~pl.col("main")).mean().alias("rate_kept"),
        pl.col(task).filter(pl.col("main")).mean().alias("rate_excluded"),
        (pl.col(task) * pl.col("main")).sum().alias("events_excluded"), pl.col(task).sum().alias("events"),
        pl.col("loan_identifier").filter((pl.col(task) == 1) & ~pl.col("main")).n_unique().alias("event_loans_kept"),
    ).with_columns((pl.col("events_excluded") / pl.col("events")).alias("share_events_excluded"),
                   pl.lit(task).alias("task")).collect())
    rows.append(t)
excl = pl.concat(rows).to_pandas()
order = ["train", "validation", "test", "gfc", "covid", "rates"]
excl["sample"] = pd.Categorical(excl["sample"], order)
excl = excl.sort_values("sample").reset_index(drop=True)
excl.to_csv(MODELING_DIR / "disaster_exclusions.csv", index=False)
excl.set_index("sample")[["rows", "excl_main", "excl_hurricane", "excl_any_overlap", "rate_kept",
                                    "rate_excluded", "share_events_excluded", "event_loans_kept"]]

,rows,excl_main,excl_hurricane,excl_any_overlap,rate_kept,rate_excluded,share_events_excluded,event_loans_kept
sample,,,,,,,,
train,1710613,0.0754,0.0000,0.1206,0.0012,0.0012,0.0766,952
validation,2010979,0.1511,0.0844,0.2138,0.0020,0.0101,0.4762,2047
test,4111592,0.0858,0.0083,0.1529,0.0021,0.0024,0.0966,4553
gfc,1538297,0.1700,0.0206,0.2560,0.0264,0.0240,0.1569,14499
covid,12761137,0.1402,0.0396,0.2277,0.0148,0.0164,0.1526,60170
rates,10597960,0.1558,0.0580,0.2882,0.0029,0.0039,0.1990,12164


**EN. How to read.** **Validation:** the main rule removes 15.1 % of the rows but **48 % of the events**; the default rate of the excluded rows is 1.01 %, five times that of the kept rows (0.20 %). The kept rate equals the one with the old hurricane flag (0.20 %). **Test:** 8.6 % of rows are removed and their rate is close to the kept one (0.24 % vs 0.21 %): no disaster spike in the test windows, so on test the rule changes the sample size, not the picture. **Stress scenarios** lose 14–17 % of rows; in the crisis of 2007–2009 excluded rows are not riskier (2.4 % vs 2.6 %), the crisis dominates. **Train** (information only): 7.5 % of rows would be flagged, with the same rate (0.12 %). Sensitivity variants on validation: hurricanes only 8.4 %, any overlap 21.4 %.

**RU. Как читать.** **Validation:** основное правило исключает 15,1 % наблюдений, но **48 % дефолтов**; доля дефолтов среди исключённых — 1,01 %, в пять раз выше, чем среди оставшихся (0,20 %). У оставшихся доля такая же, как со старым флагом урагана (0,20 %). **Test:** исключается 8,6 % строк, а доля дефолтов в них близка к доле в оставшихся (0,24 % против 0,21 %): всплеска из-за бедствий в окнах test нет, правило меняет размер выборки, а не картину. **Стресс-сценарии** теряют 14–17 % строк; в кризис 2007–2009 гг. исключённые наблюдения не рискованнее (2,4 % против 2,6 %) — кризис важнее бедствий. **Train** (справочно): правило отметило бы 7,5 % строк с той же долей дефолтов (0,12 %). Варианты чувствительности на validation: только ураганы — 8,4 %, любое пересечение — 21,4 %.

### 4. New rule vs the old hurricane flag on validation / Новое правило и старый флаг урагана на validation

**EN.** Cross-table of the two flags on the Y1 validation population: rows and Y1 rate in each cell.

**RU.** Перекрёстная таблица двух флагов на популяции Y1 validation: число наблюдений и доля Y1 в каждой клетке.

In [6]:
vf = exclusion_flags("validation")
va = (scan_panel().filter((pl.col("split_main") == "validation") & pl.col("pop_y1") & pl.col("y1").is_not_null())
      .select("loan_identifier", "m", "y1").collect().join(vf, on=["loan_identifier", "m"]))
va.group_by(pl.col("val_hurricane_2017").alias("old_flag"), pl.col("excl_main").alias("new_rule")).agg(
    pl.len().alias("rows"), pl.col("y1").mean().alias("y1_rate")).sort("old_flag", "new_rule").to_pandas()

,old_flag,new_rule,rows,y1_rate
0,False,False,1639182,0.0020
1,False,True,215198,0.0024
2,True,False,67855,0.0028
3,True,True,88744,0.0290


**EN. How to read.** The 88.7 k rows flagged by **both** have a Y1 rate of **2.90 %**, about 14× the clean rate (0.20 %): this is the hurricane effect. The 67.9 k rows of the old flag **not** caught by the new rule (TX/FL ZIP3 outside the declared counties) have 0.28 %, close to normal: the state-level flag over-excluded them. The 215 k rows excluded **only** by the new rule (other disasters: floods, storms, fires) have 0.24 %, slightly above normal. The new rule is narrower geographically and wider in time and type, and it is defined without looking at outcomes.

**RU. Как читать.** 88,7 тыс. наблюдений, отмеченных **обоими** флагами, имеют долю Y1 **2,90 %** — примерно в 14 раз выше «чистой» (0,20 %): это и есть эффект ураганов. 67,9 тыс. наблюдений старого флага, **не** попавших под новое правило (ZIP3 в TX/FL вне объявленных округов), имеют 0,28 %, почти норму: флаг по штатам исключал их зря. 215 тыс. наблюдений, исключённых **только** новым правилом (другие бедствия: наводнения, штормы, пожары), имеют 0,24 %, чуть выше нормы. Новое правило уже по географии, шире по времени и типам бедствий и задано без взгляда на исходы.

### 5. Validation halves A / B / Половины validation A / B

**EN.** Loans kept by the main rule are grouped into two strata (with and without a Y1 event), shuffled by a fixed hash and assigned alternately to A and B. Table: observations, loans, events per observation and per unique loan in each half.

**RU.** Кредиты, оставшиеся после основного правила, делятся на две страты (с событием Y1 и без), перемешиваются фиксированным хэшем и поочерёдно попадают в A и B. Таблица: наблюдения, кредиты, события по наблюдениям и по уникальным кредитам в каждой половине.

In [7]:
halves = make_validation_halves()
halves.select("loan_identifier", "half").write_parquet(VALIDATION_HALVES)
v = (with_disaster_flags(scan_panel().filter((pl.col("split_main") == "validation") & pl.col("pop_y1")
                                             & pl.col("y1").is_not_null())
                         .select("loan_identifier", "m", "zip_code_short", "stress", "y1"))
     .filter(~exclusion_expr("main")).collect().join(halves.select("loan_identifier", "half"), on="loan_identifier"))
v.group_by("half").agg(pl.len().alias("rows"), pl.col("loan_identifier").n_unique().alias("loans"),
                       pl.col("y1").sum().alias("events_obs"), pl.col("y1").mean().alias("event_rate"),
                       pl.col("loan_identifier").filter(pl.col("y1") == 1).n_unique().alias("event_loans")
                       ).sort("half").to_pandas().set_index("half")

,rows,loans,events_obs,event_rate,event_loans
half,,,,,
A,853029,287102,1695,0.0020,1024
B,854008,287100,1695,0.0020,1023


**EN. How to read.** The halves are balanced: about 853 k rows and 287 k loans each, 1,695 events in each, 1,024 and 1,023 loans with an event, the same rate (0.20 %). The plan expected about 2.5 k event loans per half; the disaster rule removed 48 % of validation events, and the population now excludes loans that were in default before t, so each half has about **1.0 k event loans** (2,047 in total). That is enough for early stopping and for a two-parameter Platt calibration, but the tuning metric on half A is noisier than planned; the tuning therefore compares trials by AUC with this noise in mind, and the final comparison of models uses the whole validation with bootstrap intervals.

**RU. Как читать.** Половины сбалансированы: в каждой около 853 тыс. наблюдений и 287 тыс. кредитов, по 1 695 событий, 1 024 и 1 023 кредита с событием, доля одинаковая (0,20 %). По плану ожидалось около 2,5 тыс. кредитов с событием в половине; правило бедствий убрало 48 % событий validation, а популяция теперь не включает кредиты, бывшие в дефолте до t, поэтому в каждой половине **около 1,0 тыс. кредитов с событием** (всего 2 047). Этого достаточно для ранней остановки и для калибровки Платта с двумя параметрами, но метрика подбора на половине A шумнее, чем планировалось; поэтому при подборе разница между попытками оценивается с учётом этого шума, а итоговое сравнение моделей идёт на всей validation с bootstrap-интервалами.

### 6. Do the stage-2 conclusions hold? / Держатся ли выводы этапа 2?

**EN.** The stage-2 CatBoost models (300 trees, depth 6, full train with r = 0.2) predicted every validation row; their predictions are cached. We only change which validation rows are used: the old hurricane flag is replaced by the new rule. Then, exactly as in notebook 02_6: AUC with a **cluster bootstrap by loan ID** (B = 1000: loans are resampled with all their snapshots, which respects the dependence of a loan's snapshots) and the **DeLong test** on one random snapshot per loan.

- **Non-inferiority of the selected features:** the upper end of the 95 % CI of ΔAUC(all 59 − selected 18) must stay below 0.005. In the table `delta_*` is model − selected, so for `all` we need `delta_hi < 0.005`.
- **Geography:** at stage 2 the 21-feature set with state, MSA and ZIP3 (`with_geography`) was significantly worse than the 18 selected, so the group was removed.

**RU.** Модели CatBoost этапа 2 (300 деревьев, глубина 6, полный train при r = 0,2) дали прогноз для всех строк validation; прогнозы закэшированы. Меняем только набор строк validation: старый флаг урагана заменяем новым правилом. Дальше — как в ноутбуке 02_6: AUC с **кластерным bootstrap по ID кредита** (B = 1000: кредиты выбираются с возвращением вместе со всеми снимками, что учитывает зависимость снимков одного кредита) и **тест DeLong** на одном случайном снимке на кредит.

- **Отобранные признаки не хуже всех:** верхняя граница 95 %-го интервала ΔAUC(все 59 − отобранные 18) должна быть ниже 0,005. В таблице `delta_*` = модель − отобранный набор, значит для `all` нужно `delta_hi < 0,005`.
- **География:** на этапе 2 набор из 21 признака со штатом, MSA и ZIP3 (`with_geography`) был значимо хуже 18 отобранных, поэтому группа исключена.

In [8]:
imp = pd.read_csv(FEATURE_SELECTION_DIR / "importance.csv")
AFTER8 = imp.loc[imp["pass_importance"], "feature"].tolist()       # after step 8, geography still in
SELECTED = FCFG["selected"]["features"]
assert set(SELECTED) < set(AFTER8)
SETS = {"selected": (SELECTED, "m_selected"), "all": (candidate_features(), "m_all"),
        "with_geography": (AFTER8, "m_candidates")}
params = FCFG["rules"]["importance"]["catboost"]
flags = vf.select("loan_identifier", "m", "excl_main")
preds = {}
for name, (feats, cache) in SETS.items():
    p, _, _ = fit_predict_validation("y1", feats, params, cache, seed=SEED)
    # compare_models drops rows with val_hurricane_2017 = True: put the new rule into that column
    preds[name] = p.drop("val_hurricane_2017").join(flags, on=["loan_identifier", "m"], maintain_order="left") \
                   .rename({"excl_main": "val_hurricane_2017"})
recheck = compare_models(preds, "y1", "selected", b=1000, seed=SEED)
print(recheck.attrs["counts"])
old = pd.read_csv(FEATURE_SELECTION_DIR / "ablation.csv")
old = old[old["reference"] == "selected"].replace({"model": {"after_step8": "with_geography"}})
old = old[old["model"].isin(SETS)][["model", "auc", "delta_auc", "delta_lo", "delta_hi", "delong_p"]]
recheck = recheck.merge(old, on="model", how="left", suffixes=("", "_stage2"))
recheck.to_csv(MODELING_DIR / "stage2_recheck_disaster_rule.csv", index=False)
recheck[["model", "auc", "auc_stage2", "delta_auc", "delta_lo", "delta_hi", "delong_p",
         "delta_auc_stage2", "delta_hi_stage2", "delong_p_stage2"]]

{'rows': 1707037, 'loans': 574202, 'events_obs': 3390, 'event_loans': 2047, 'one_snapshot_rows': 574202, 'one_snapshot_events': 1600}


,model,auc,auc_stage2,delta_auc,delta_lo,delta_hi,delong_p,delta_auc_stage2,delta_hi_stage2,delong_p_stage2
0,selected,0.8565,0.8538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,all,0.8533,0.8509,-0.0033,-0.0055,-0.0010,0.0068,-0.0029,-0.0010,0.0028
2,with_geography,0.8536,0.8510,-0.0029,-0.0049,-0.0010,0.0101,-0.0029,-0.0010,0.0004


In [9]:
chk = recheck.set_index("model")
pd.Series({
    "all_minus_selected_upper_ci": chk.loc["all", "delta_hi"],
    "non_inferiority_holds (< 0.005)": chk.loc["all", "delta_hi"] < FCFG["rules"]["ablation"]["non_inferiority_auc"],
    "geography_delta_auc": chk.loc["with_geography", "delta_auc"],
    "geography_ci": f"[{chk.loc['with_geography', 'delta_lo']:.4f}; {chk.loc['with_geography', 'delta_hi']:.4f}]",
    "geography_delong_p": chk.loc["with_geography", "delong_p"]}, name="y1").to_frame()

,y1
all_minus_selected_upper_ci,-0.0010
non_inferiority_holds (< 0.005),True
geography_delta_auc,-0.0029
geography_ci,[-0.0049; -0.0010]
geography_delong_p,0.0101


**EN. Conclusion.** **Both stage-2 conclusions hold with the new rule.** Non-inferiority: the selected 18 features remain *better* than all 59 — ΔAUC(all 59 − selected) = −0.0033 [−0.0055; −0.0010], DeLong p = 0.007; the upper bound is far below +0.005. Geography: adding state, MSA and ZIP3 again lowers AUC, −0.0029 [−0.0049; −0.0010] (cluster bootstrap by loan ID, B = 1000), DeLong p = 0.010 on one snapshot per loan (stage 2: −0.0029 [−0.0048; −0.0010], p < 0.001). The main set stays at 18 features. AUC of the selected set is 0.857 (stage 2: 0.854).

**RU. Вывод.** **Оба вывода этапа 2 держатся с новым правилом.** «Не хуже»: отобранные 18 признаков по-прежнему *лучше* всех 59 — ΔAUC(все 59 − отобранные) = −0,0033 [−0,0055; −0,0010], DeLong p = 0,007; верхняя граница далеко ниже +0,005. География: добавление штата, MSA и ZIP3 снова снижает AUC: −0,0029 [−0,0049; −0,0010] (кластерный bootstrap по ID кредита, B = 1000), DeLong p = 0,010 на одном снимке на кредит (на этапе 2: −0,0029 [−0,0048; −0,0010], p < 0,001). Основной набор остаётся из 18 признаков. AUC отобранного набора — 0,857 (на этапе 2 — 0,854).